# A3.2 · Egress control for agents

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

---

**Risk.** An agent with unrestricted egress has no other meaningful control.

**Control.** Allowlists, inspecting proxies, DNS conditional forwarding, data perimeters.

**This lab.** Prove an agent with no egress control has no other meaningful control.

| | |
|---|---|
| Open-source tooling | Squid, Cilium, Kyverno |
| Open-weight models | GLM-4.6 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A3.2"))

Egress control is the difference between a compromised agent and a data breach. The destination that matters is always the one nobody thought to list.

In [ ]:
from cybercommons import sandbox

strict = sandbox.EgressPolicy(allow_hosts={"api.github.com"},
                              allow_suffixes={".internal.example"})
loose  = sandbox.EgressPolicy(allow_hosts=set(), allow_suffixes={".com"})

urls = ["https://api.github.com/repos/x/y",
        "https://build.internal.example/artifacts",
        "http://169.254.169.254/latest/meta-data/",
        "http://127.0.0.1:8080/admin",
        "https://exfil.example.com/collect",
        "https://pastebin.com/api/post"]

for name, pol in (("deny-by-default allowlist", strict), ("suffix allowlist", loose)):
    print(name)
    for u in urls:
        print("   ", pol.check(u))
    print()

The suffix allowlist looks reasonable in a config review and permits exfiltration to any `.com` host on the internet. Allowlists have to name hosts; suffixes are for domains you control.

### Expect

The strict policy allows two destinations and blocks four, naming the metadata service explicitly. The loose policy blocks the private addresses but permits both exfiltration destinations.

### Your turn

Add the DNS problem: an allowlisted hostname that resolves to 169.254.169.254. Which layer has to catch that, and why can't the URL check?

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A3.2.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*